# Audit Hypothesis Agent — Multi-Agent (LangGraph)

Расширение `rag_system_v2.ipynb`: добавляет мультиагентный граф с циклом самопроверки.

## Архитектура графа

```
START
  │
  ▼
[Planner]      — декомпозирует вопрос на 2-3 фокусированных под-вопроса
  │
  ▼
[Retriever]    — параллельный поиск по всем под-вопросам + reranking
  │
  ▼
[Generator]    — генерирует первичные гипотезы
  │
  ▼
[Critic]       — оценивает качество (0-10), выявляет пробелы
  │
  ├─ score < 7 и iterations < 2 ──► [Refiner] ──► [Critic]  (цикл)
  │
  └─ score ≥ 7 или max_iter ──────► [Reporter]
                                        │
                                       END
```

## Что добавляет каждый агент

| Агент | LLM-вызов | Что делает |
|-------|-----------|------------|
| Planner | ~150 токенов | Разбивает вопрос на под-темы → лучший recall |
| Retriever | нет | Поиск + reranking по всем под-темам |
| Generator | ~2000 токенов | Первичные гипотезы |
| Critic | ~600 токенов | Оценка 0-10 + конкретные замечания |
| Refiner | ~1500 токенов | Улучшение по замечаниям Critic |
| Reporter | нет | Форматирование в HypothesesReport |

**Итого:** 3-5 LLM-вызовов против 2 в v2. На GPU (4-bit) — приемлемо. На CPU — медленно.

## 1. Установка LangGraph

In [ ]:
%pip install -q langgraph langchain-core

## 2. Запуск rag_system_v2

Перед этим ноутбуком должен быть выполнен `rag_system_v2.ipynb`.
Все классы (DocumentRetriever, AuditHypothesis, AuditAgent и т.д.) переиспользуются.

In [ ]:
import os, re, torch
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict, Optional

# Проверяем, что нужные объекты загружены из rag_system_v2
for name in ("agent", "model", "tokenizer", "reranker",
             "parse_hypotheses", "HypothesesReport"):
    assert name in dir(), f"'{name}' не найден — запусти сначала rag_system_v2.ipynb"

from langgraph.graph import StateGraph, END
print("LangGraph готов ✓")


## 3. State — общее состояние графа

In [ ]:
class AuditState(TypedDict):
    """Состояние, передаваемое между всеми агентами графа."""
    # Входные данные
    question: str
    max_iterations: int

    # Planner
    sub_questions: List[str]

    # Retriever
    chunks: List[Dict]

    # Generator / Refiner
    raw_hypotheses: str
    needs_regen: bool       # Verifier запросил повторную генерацию

    # Critic
    critique: str
    quality_score: int      # 0-10, порог для рефайна = 7
    iterations: int

    # Verifier
    grounding_score: float  # 0.0-1.0, доля подтверждённых гипотез
    regen_iterations: int   # счётчик повторных генераций

    # Reporter
    final_report: Optional[Dict]


## 4. Вспомогательная функция генерации

In [ ]:
def llm_call(system: str, user: str,
             max_new_tokens: int = 512,
             temperature: float = 0.4) -> str:
    """Единая точка вызова LLM для всех агентов графа."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

## 5. Узлы графа (агенты)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# PLANNER — декомпозирует вопрос на 2-3 под-темы
# ─────────────────────────────────────────────────────────────────────
def planner_node(state: AuditState) -> dict:
    print("[Planner] Декомпозиция вопроса...")
    raw = llm_call(
        system="Ты — опытный аудитор банковской группы.",
        user=(
            f"Разбей следующий аудиторский вопрос на 2-3 узкие под-темы "
            f"для более точного поиска по документам.\n"
            f"Каждую под-тему напиши на отдельной строке. Без нумерации и пояснений.\n\n"
            f"Вопрос: {state['question']}"
        ),
        max_new_tokens=200, temperature=0.3,
    )
    sub_questions = [state["question"]] + [
        l.strip() for l in raw.splitlines() if l.strip()
    ][:3]
    print(f"  Под-вопросов: {len(sub_questions)}")
    for q in sub_questions[1:]:
        print(f"  • {q}")
    return {"sub_questions": sub_questions}


# ─────────────────────────────────────────────────────────────────────
# RETRIEVER — поиск по всем под-темам + CrossEncoder reranking
# ─────────────────────────────────────────────────────────────────────
def retriever_node(state: AuditState) -> dict:
    print("[Retriever] Поиск по всем под-вопросам...")
    all_results = []
    for q in state["sub_questions"]:
        all_results.extend(agent.retriever.search(q, top_k=6))
    unique = list({r["text"]: r for r in all_results}.values())
    if not unique:
        print("  ⚠ База знаний пуста")
        return {"chunks": []}
    pairs = [[state["question"], r["text"]] for r in unique]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(unique, scores), key=lambda x: x[1], reverse=True)
    chunks = [r for r, _ in ranked[:8]]
    print(f"  Отобрано фрагментов: {len(chunks)}")
    return {"chunks": chunks}


# ─────────────────────────────────────────────────────────────────────
# GENERATOR — первичные гипотезы
# При needs_regen=True добавляет явное требование опираться на источники
# ─────────────────────────────────────────────────────────────────────
def generator_node(state: AuditState) -> dict:
    print("[Generator] Генерация гипотез...")
    context = agent._build_context(state["chunks"])
    prompt  = agent._build_prompt(context, state["question"])
    if state.get("needs_regen"):
        prompt = (
            "ВАЖНО: предыдущие гипотезы не были подтверждены источниками. "
            "Генерируй ТОЛЬКО то, что явно следует из текстов ниже.\n\n" + prompt
        )
    raw = llm_call(
        system="Ты — ассистент ведущего аудитора банковской группы.",
        user=prompt, max_new_tokens=2048, temperature=0.6,
    )
    print(f"  Сгенерировано {len(raw)} символов")
    return {"raw_hypotheses": raw, "iterations": 0, "needs_regen": False}


# ─────────────────────────────────────────────────────────────────────
# CRITIC — оценивает качество гипотез 0-10
# ─────────────────────────────────────────────────────────────────────
def critic_node(state: AuditState) -> dict:
    iteration = state.get("iterations", 0) + 1
    print(f"[Critic] Оценка качества (итерация {iteration})...")
    raw = llm_call(
        system="Ты — строгий эксперт по аудиту. Оцениваешь качество аудиторских гипотез.",
        user=(
            f"Оцени гипотезы по трём критериям:\n"
            f"1. Конкретность: указаны конкретные риски, не общие фразы?\n"
            f"2. Проверяемость: есть конкретные шаги проверки?\n"
            f"3. Полнота: охвачены ключевые риски из контекста?\n\n"
            f"Формат ответа:\n"
            f"ОЦЕНКА: [0-10]\n"
            f"ЗАМЕЧАНИЯ:\n- ...\n\n"
            f"Вопрос: {state['question']}\n\n"
            f"Гипотезы:\n{state['raw_hypotheses'][:3000]}"
        ),
        max_new_tokens=600, temperature=0.3,
    )
    score_match = re.search(r"ОЦЕНКА:\s*(\d+)", raw)
    quality_score = max(0, min(10, int(score_match.group(1)))) if score_match else 5
    print(f"  Оценка: {quality_score}/10")
    return {"critique": raw, "quality_score": quality_score, "iterations": iteration}


# ─────────────────────────────────────────────────────────────────────
# REFINER — дорабатывает гипотезы по замечаниям Critic
# ─────────────────────────────────────────────────────────────────────
def refiner_node(state: AuditState) -> dict:
    print("[Refiner] Доработка гипотез...")
    context = agent._build_context(state["chunks"])
    raw = llm_call(
        system="Ты — ассистент ведущего аудитора банковской группы.",
        user=(
            f"Улучши гипотезы по замечаниям эксперта.\n\n"
            f"ЗАМЕЧАНИЯ:\n{state['critique']}\n\n"
            f"ИСХОДНЫЕ ГИПОТЕЗЫ:\n{state['raw_hypotheses'][:2000]}\n\n"
            f"КОНТЕКСТ:\n{context[:4000]}\n\n"
            f"Вопрос: {state['question']}\n\n"
            f"Сохрани хорошие гипотезы, конкретизируй слабые, "
            f"добавь шаги проверки, укажи уровень риска HIGH/MEDIUM/LOW."
        ),
        max_new_tokens=2048, temperature=0.5,
    )
    print(f"  Доработано: {len(raw)} символов")
    return {"raw_hypotheses": raw}


# ─────────────────────────────────────────────────────────────────────
# VERIFIER — проверяет каждую гипотезу на наличие подтверждения
#
# Для каждой гипотезы:
#   → ищет релевантные чанки через CrossEncoder (без LLM)
#   → задаёт LLM один вопрос: «подтверждается ли?»
# Добавляет метки [Подтверждено: ИСТОЧНИК N] / [НЕ ПОДТВЕРЖДЕНО]
# Если grounding_score < 0.5 — запрашивает повторную генерацию
# ─────────────────────────────────────────────────────────────────────
def verifier_node(state: AuditState) -> dict:
    print("[Verifier] Проверка заземления гипотез на источниках...")

    if not state["chunks"]:
        print("  ⚠ Нет источников для верификации")
        return {"grounding_score": 0.0, "needs_regen": False}

    context = agent._build_context(state["chunks"])

    raw = llm_call(
        system=(
            "Ты — строгий аудитор. Проверяешь, подтверждается ли каждая гипотеза "
            "предоставленными документами. Отвечаешь только на основе текста источников."
        ),
        user=(
            f"Для каждой гипотезы добавь метку в конце:\n"
            f"  [Подтверждено: ИСТОЧНИК N] — если источник явно подтверждает\n"
            f"  [Частично подтверждено: ИСТОЧНИК N] — если косвенно\n"
            f"  [НЕ ПОДТВЕРЖДЕНО ИСТОЧНИКАМИ] — если источников нет\n\n"
            f"Правила:\n"
            f"- Не меняй текст гипотез, только добавляй метки в конец каждой\n"
            f"- Не придумывай подтверждения — только то, что есть в источниках\n\n"
            f"ИСТОЧНИКИ:\n{context[:5000]}\n\n"
            f"ГИПОТЕЗЫ:\n{state['raw_hypotheses'][:3000]}"
        ),
        max_new_tokens=2500, temperature=0.2,
    )

    # Считаем grounding_score
    unverified = raw.count("[НЕ ПОДТВЕРЖДЕНО")
    confirmed  = raw.count("[Подтверждено") + raw.count("[Частично")
    total      = max(unverified + confirmed, 1)
    grounding_score = confirmed / total
    print(f"  Подтверждено: {confirmed}/{total}  (grounding={grounding_score:.2f})")

    regen_iter = state.get("regen_iterations", 0)
    needs_regen = grounding_score < 0.5 and regen_iter < 1
    if needs_regen:
        print(f"  ⚠ grounding < 0.5 — запрашиваем повторную генерацию (попытка {regen_iter+1})")

    return {
        "raw_hypotheses": raw,
        "grounding_score": grounding_score,
        "needs_regen": needs_regen,
        "regen_iterations": regen_iter + 1,
    }


# ─────────────────────────────────────────────────────────────────────
# REPORTER — форматирует финальный HypothesesReport
# ─────────────────────────────────────────────────────────────────────
def reporter_node(state: AuditState) -> dict:
    print("[Reporter] Формирование отчёта...")
    report = parse_hypotheses(state["raw_hypotheses"], state["question"], state["chunks"])
    print(f"  Гипотез: {len(report.hypotheses)}, источников: {report.sources_used}")
    return {"final_report": report.dict()}


## 6. Условное ребро: рефайн или финиш?

In [ ]:
def should_refine_or_verify(state: AuditState) -> str:
    """После Critic: рефайн или верификация?"""
    score    = state.get("quality_score", 0)
    iters    = state.get("iterations", 0)
    max_iter = state.get("max_iterations", 2)
    if score < 7 and iters < max_iter:
        print(f"  ↻ {score}/10 < 7, итерация {iters}/{max_iter} → Refiner")
        return "refine"
    print(f"  ✓ {score}/10, итерация {iters}/{max_iter} → Verifier")
    return "verify"


def should_regen_or_report(state: AuditState) -> str:
    """После Verifier: регенерация или финальный отчёт?"""
    if state.get("needs_regen", False):
        print("  ↻ Низкий grounding → Generator")
        return "regen"
    print(f"  ✓ grounding={state.get('grounding_score', 0):.2f} → Reporter")
    return "report"


## 7. Сборка графа

In [ ]:
workflow = StateGraph(AuditState)

workflow.add_node("planner",   planner_node)
workflow.add_node("retriever", retriever_node)
workflow.add_node("generator", generator_node)
workflow.add_node("critic",    critic_node)
workflow.add_node("refiner",   refiner_node)
workflow.add_node("verifier",  verifier_node)
workflow.add_node("reporter",  reporter_node)

workflow.set_entry_point("planner")
workflow.add_edge("planner",   "retriever")
workflow.add_edge("retriever", "generator")
workflow.add_edge("generator", "critic")
workflow.add_edge("refiner",   "critic")

# Critic → Refiner (если score < 7) или Verifier
workflow.add_conditional_edges(
    "critic", should_refine_or_verify,
    {"refine": "refiner", "verify": "verifier"},
)

# Verifier → Generator (если grounding < 0.5) или Reporter
workflow.add_conditional_edges(
    "verifier", should_regen_or_report,
    {"regen": "generator", "report": "reporter"},
)

workflow.add_edge("reporter", END)

audit_graph = workflow.compile()
print("Граф скомпилирован ✓")
print()
print("Структура графа:")
print("  planner → retriever → generator → critic")
print("                              ↑         │ score < 7")
print("                              │      refiner")
print("                              │         │ score ≥ 7")
print("                              │      verifier")
print("                     grounding < 0.5 ↙   ↘ grounding ≥ 0.5")
print("                           (regen)      reporter → END")

try:
    from IPython.display import Image, display
    display(Image(audit_graph.get_graph().draw_mermaid_png()))
except Exception:
    pass


## 8. Запуск

In [ ]:
question = (
    "Вычитка информации о финансовых транзакциях по бизнес-карте "
    "корпоративного клиента и их отображение по счёту клиента"
)

initial_state: AuditState = {
    "question":         question,
    "max_iterations":   2,
    "sub_questions":    [],
    "chunks":           [],
    "raw_hypotheses":   "",
    "needs_regen":      False,
    "critique":         "",
    "quality_score":    0,
    "iterations":       0,
    "grounding_score":  0.0,
    "regen_iterations": 0,
    "final_report":     None,
}

print("=" * 60)
print(f"Вопрос: {question}")
print("=" * 60)

final_state = audit_graph.invoke(initial_state)


## 9. Вывод результатов

In [ ]:
report = HypothesesReport(**final_state["final_report"])

print("\n" + "=" * 60)
print(report.to_markdown())

print("\n" + "=" * 60)
print(f"Итоговая оценка Critic    : {final_state['quality_score']}/10")
print(f"Grounding score (верифик.): {final_state['grounding_score']:.2f}")
print(f"Итераций Critic→Refiner   : {final_state['iterations']}")
print(f"Итераций Verifier→Regen   : {final_state['regen_iterations']}")
print(f"Под-вопросов (Planner)    : {len(final_state['sub_questions'])}")
print(f"Фрагментов использовано   : {report.sources_used}")
if report.source_names:
    print("Источники:")
    for s in report.source_names:
        print(f"  • {s}")


## 10. Потоковый вывод (stream)

LangGraph позволяет смотреть результат каждого агента по мере выполнения.

In [ ]:
# Потоковый запуск — смотреть результат каждого агента в реальном времени
# Запускай ВМЕСТО ячейки lg-run (не после), иначе граф выполнится дважды

print("Запуск с потоковым выводом...\n")

for step in audit_graph.stream(initial_state):
    node_name = list(step.keys())[0]
    node_out  = step[node_name]
    print(f"\n── {node_name.upper()} ──")
    for key, val in node_out.items():
        if key in ("raw_hypotheses", "critique"):
            print(f"  {key}: {str(val)[:300]}...")
        elif key == "chunks":
            print(f"  chunks: {len(val)} фрагментов")
        elif key == "final_report":
            n = len(val.get("hypotheses", [])) if val else 0
            print(f"  final_report: {n} гипотез")
        else:
            print(f"  {key}: {val}")
